# DBLP AI Author Coauthor Graph

This notebook runs the DBLP coauthor graph pipeline step by step.

It reuses the implementation from `scripts/fetch_dblp_ai_coauthor_graph.py` and exports:

- `papers.csv`
- `authors.csv`
- `edges.csv`
- `graph.graphml`
- `summary.json`

The default definition is:

`AI author = an author who published at least one paper in AAAI, IJCAI, ICML, NeurIPS, or ICLR during the selected year range`

## 1. Configuration

Edit these values before running the pipeline. DBLP venue key `nips` corresponds to NeurIPS.

In [4]:
from __future__ import annotations

import json
from argparse import Namespace
from collections import Counter
from datetime import datetime
from pathlib import Path

from scripts.fetch_dblp_ai_coauthor_graph import (
    DBLP_BASE_URL,
    DEFAULT_VENUES,
    build_edge_rows,
    collect_dataset,
    write_csv,
    write_graphml,
)

START_YEAR = 2015
END_YEAR = 2025
VENUES = ["icml", "nips", "iclr"]

SLEEP_SECONDS = 10
TIMEOUT_SECONDS = 60.0
MAX_RETRIES = 6

OUTPUT_DIR = Path(f"data/dblp_ai_authors_{START_YEAR}_{END_YEAR}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR

PosixPath('data/dblp_ai_authors_2015_2025')

## 2. Validate Settings

In [5]:
venues = [venue.lower() for venue in VENUES]

if START_YEAR > END_YEAR:
    raise ValueError("START_YEAR must be <= END_YEAR")

invalid = [venue for venue in venues if venue not in DEFAULT_VENUES]
if invalid:
    supported = ", ".join(DEFAULT_VENUES)
    raise ValueError(f"Unsupported venues: {', '.join(invalid)}. Supported: {supported}")

config = {
    "venues": {venue: DEFAULT_VENUES[venue] for venue in venues},
    "year_range": {"start": START_YEAR, "end": END_YEAR},
    "output_dir": str(OUTPUT_DIR),
    "sleep_seconds": SLEEP_SECONDS,
    "timeout_seconds": TIMEOUT_SECONDS,
    "max_retries": MAX_RETRIES,
}
config

{'venues': {'icml': 'ICML', 'nips': 'NeurIPS', 'iclr': 'ICLR'},
 'year_range': {'start': 2015, 'end': 2025},
 'output_dir': 'data/dblp_ai_authors_2015_2025',
 'sleep_seconds': 10,
 'timeout_seconds': 60.0,
 'max_retries': 6}

## 3. Fetch DBLP Papers

This fetches each per-year proceedings record, follows it to the DBLP table-of-contents XML, and extracts paper/authorship metadata.

In [6]:
args = Namespace(
    start_year=START_YEAR,
    end_year=END_YEAR,
    venues=venues,
    output_dir=OUTPUT_DIR,
    sleep_seconds=SLEEP_SECONDS,
    timeout_seconds=TIMEOUT_SECONDS,
    max_retries=MAX_RETRIES,
)

papers, authors, venue_counts = collect_dataset(args)

{
    "papers": len(papers),
    "authors": len(authors),
    "paper_counts_by_venue": dict(venue_counts),
}

[venue] ICML (icml)
[retry] curl failed for https://dblp.org/rec/conf/icml/2015.xml: curl: (56) Recv failure: Operation timed out -> sleeping 10.0s
[retry] curl failed for https://dblp.org/rec/conf/icml/2015.xml: curl: (56) Recv failure: Operation timed out -> sleeping 20.0s
  - 2015: fetching https://dblp.org/db/conf/icml/icml2015.xml
    found 270 papers
  - 2016: fetching https://dblp.org/db/conf/icml/icml2016.xml
    found 322 papers
  - 2017: fetching https://dblp.org/db/conf/icml/icml2017.xml
    found 434 papers
  - 2018: fetching https://dblp.org/db/conf/icml/icml2018.xml
    found 621 papers
  - 2019: fetching https://dblp.org/db/conf/icml/icml2019.xml
    found 773 papers
  - 2020: fetching https://dblp.org/db/conf/icml/icml2020.xml
    found 1085 papers
  - 2021: fetching https://dblp.org/db/conf/icml/icml2021.xml
    found 1183 papers
  - 2022: fetching https://dblp.org/db/conf/icml/icml2022.xml
    found 1234 papers
  - 2023: fetching https://dblp.org/db/conf/icml/icml2023

{'papers': 44123,
 'authors': 67224,
 'paper_counts_by_venue': {'ICML': 13617, 'NeurIPS': 19181, 'ICLR': 11325}}

## 4. Build Graph Tables

In [7]:
edge_rows = build_edge_rows(papers)

author_rows = []
for author in sorted(authors.values(), key=lambda row: (-len(row["paper_ids"]), row["name"])):
    author_rows.append(
        {
            "author_id": author["author_id"],
            "name": author["name"],
            "dblp_pid": author["dblp_pid"],
            "paper_count": len(author["paper_ids"]),
            "paper_ids": "|".join(author["paper_ids"]),
            "venues": "|".join(sorted(author["venues"])),
            "years": "|".join(str(year) for year in sorted(author["years"])),
        }
    )

paper_rows = [
    {
        "paper_id": paper.paper_id,
        "dblp_key": paper.dblp_key,
        "title": paper.title,
        "year": paper.year,
        "venue_key": paper.venue_key,
        "venue_name": paper.venue_name,
        "booktitle": paper.booktitle,
        "ee": paper.ee,
        "pages": paper.pages,
        "crossref": paper.crossref,
        "dblp_url": paper.dblp_url,
        "toc_url": paper.toc_url,
        "author_ids": "|".join(paper.author_ids),
        "author_names": "|".join(paper.author_names),
        "author_count": len(paper.author_ids),
    }
    for paper in papers
]

{
    "paper_rows": len(paper_rows),
    "author_rows": len(author_rows),
    "edge_rows": len(edge_rows),
}

{'paper_rows': 44123, 'author_rows': 67224, 'edge_rows': 407230}

## 5. Export Files

In [8]:
write_csv(
    OUTPUT_DIR / "papers.csv",
    paper_rows,
    [
        "paper_id",
        "dblp_key",
        "title",
        "year",
        "venue_key",
        "venue_name",
        "booktitle",
        "ee",
        "pages",
        "crossref",
        "dblp_url",
        "toc_url",
        "author_ids",
        "author_names",
        "author_count",
    ],
)

write_csv(
    OUTPUT_DIR / "authors.csv",
    author_rows,
    [
        "author_id",
        "name",
        "dblp_pid",
        "paper_count",
        "paper_ids",
        "venues",
        "years",
    ],
)

write_csv(
    OUTPUT_DIR / "edges.csv",
    edge_rows,
    [
        "source_author_id",
        "target_author_id",
        "weight",
        "paper_count",
        "paper_ids",
    ],
)

write_graphml(OUTPUT_DIR / "graph.graphml", author_rows, edge_rows)

summary = {
    "generated_at": datetime.now().isoformat(),
    "source": "DBLP proceedings XML",
    "dblp_base_url": DBLP_BASE_URL,
    "venues": {venue: DEFAULT_VENUES[venue] for venue in venues},
    "year_range": {"start": START_YEAR, "end": END_YEAR},
    "counts": {
        "papers": len(paper_rows),
        "authors": len(author_rows),
        "edges": len(edge_rows),
    },
    "paper_counts_by_venue": dict(venue_counts),
    "notes": [
        "DBLP venue streams provide metadata only; paper-level records are fetched from per-year proceedings TOC XML.",
        "Author identity uses DBLP pid when available, otherwise a normalized name key.",
        "The output graph is an undirected weighted coauthor graph where edge weight equals the number of qualifying papers.",
    ],
}

(OUTPUT_DIR / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

summary

{'generated_at': '2026-04-16T13:50:56.801626',
 'source': 'DBLP proceedings XML',
 'dblp_base_url': 'https://dblp.org/',
 'venues': {'icml': 'ICML', 'nips': 'NeurIPS', 'iclr': 'ICLR'},
 'year_range': {'start': 2015, 'end': 2025},
 'counts': {'papers': 44123, 'authors': 67224, 'edges': 407230},
 'paper_counts_by_venue': {'ICML': 13617, 'NeurIPS': 19181, 'ICLR': 11325},
 'notes': ['DBLP venue streams provide metadata only; paper-level records are fetched from per-year proceedings TOC XML.',
  'Author identity uses DBLP pid when available, otherwise a normalized name key.',
  'The output graph is an undirected weighted coauthor graph where edge weight equals the number of qualifying papers.']}

## 6. Quick Checks

In [9]:
top_authors = author_rows[:20]
counts_by_year = Counter(paper["year"] for paper in paper_rows)
counts_by_venue_year = Counter((paper["venue_name"], paper["year"]) for paper in paper_rows)

{
    "top_authors": top_authors,
    "counts_by_year": dict(sorted(counts_by_year.items())),
    "counts_by_venue_year": {
        f"{venue} {year}": count
        for (venue, year), count in sorted(counts_by_venue_year.items())
    },
    "output_files": sorted(path.name for path in OUTPUT_DIR.iterdir()),
}

{'top_authors': [{'author_id': 'pid:80/7594',
   'name': 'Sergey Levine',
   'dblp_pid': '80/7594',
   'paper_count': 248,
   'paper_ids': 'conf-icml-schulmanlajm15|conf-icml-finnla16|conf-icml-gulsl16|conf-icml-andreaskl17|conf-icml-chebotarhzssl17|conf-icml-finnal17|conf-icml-haarnojatal17|conf-icml-co-reyeslgeal18|conf-icml-haarnojahal18|conf-icml-haarnojazal18|conf-icml-jinkl18a|conf-icml-srinivasjalf18|conf-icml-tuckerbgtgl18|conf-icml-finnrkl19|conf-icml-fuksl19|conf-icml-kimkjls19|conf-icml-rakellyzflq19|conf-icml-xurdlf19|conf-icml-zhangvsa0l19|conf-icml-0003kwgl20|conf-icml-filostmrlg20|conf-icml-pongdlnbl20|conf-icml-reddydlll20|conf-icml-zhangcflj20|conf-icml-0003klg21|conf-icml-chebotarhlxkvie21|conf-icml-choisllg21|conf-icml-filoslgljf21|conf-icml-furutamkmlng21|conf-icml-kohsmxzbhypglds21|conf-icml-li0rpzyl21|conf-icml-mitchellrplf21|conf-icml-ndousseelj21|conf-icml-rybkindl21|conf-icml-rybkinzndml21|conf-icml-trabuccokgl21|conf-icml-zhoul21|conf-icml-ghoshaal22|conf-icml